# Review saved forecasts and evaluations

Read-only charts and tables. Run after 06; actual/error overlays require completed 07 evaluations. The comparison uses saved forecasts and frozen actuals to explain accuracy and coverage.

Set CONFIG_PATH to the same shared YAML used by 05/06/07, then run all cells. Choose the view, experiment, game with separate Sale/Rent choices, and forecast run. The default view includes only evaluated test games. Choose the explicit forecast-only/unevaluated view for inference targets.

Actual overlays come from the selected forecast's latest completed evaluation cohort. Missing or insufficient labels remain visible as eligibility reasons; they are not inserted into accuracy metrics. Only selected curves/neighbors/metrics are collected for plotting. Earlier saved evaluations remain available in the result tables.

After the target forecast, **Like games and historical results** shows the retained Sale or Rent list alongside the Eilers list for the selected run. Each game has a commercial and performance history wherever its saved names match the source data. Expand the monthly tables for exact values and calendar dates. The charts use observed months from the source versions recorded by 06.

The **Compare approaches and inspect failures** section at the end compares saved evaluations for Sales, Rent, and the two Eilers indices. Start with Validation, then check Held-out test.

Use the demand curve On/Off buttons to overlay the neighbor mean, linear, quadratic and logarithmic forecasts. Observed actuals stay visible for comparison. The per-game table identifies each saved score by curve.

The experiment dropdown uses the YAML experiment name. Each named forecast run shows its demand rank weights. Earlier saved forecasts are also available.


In [0]:
CONFIG_PATH = "configs/experiments.yaml"


In [0]:
"""Validated YAML configuration and semantic work identities for notebook 05."""
import copy
import hashlib
import json
import math
import re
from datetime import date, datetime
from numbers import Integral, Real
from pathlib import Path

SIMILARITY_IMPLEMENTATION_VERSION = "05_batch_yaml_v1_2026_11_19"
WEIGHT_GROUPS = ("mechanics", "themes", "reels", "persistence", "ways", "months_since_release")
PARENT_PROCESSES = {"p1_parent_games_only", "p2_current", "p3_include_parent"}


def config_json(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False,
                      default=lambda x: x.isoformat() if isinstance(x, (date, datetime)) else str(x))


def config_sha256(value):
    return hashlib.sha256(config_json(value).encode("utf-8")).hexdigest()


def quoted_table(name):
    return ".".join("`" + p.replace("`", "``") + "`" for p in name.split("."))


def _keys(value, allowed, where, required=()):
    if not isinstance(value, dict):
        raise ValueError(f"{where} must be a mapping.")
    unknown, missing = set(value) - set(allowed), set(required) - set(value)
    if unknown or missing:
        raise ValueError(f"{where}: unknown keys={sorted(unknown)}, missing keys={sorted(missing)}")


def _text(value, where):
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"{where} must be a nonempty string.")
    return value.strip()


def _integer(value, where, minimum=0):
    if isinstance(value, bool) or not isinstance(value, Integral) or value < minimum:
        raise ValueError(f"{where} must be an integer >= {minimum}.")
    return int(value)


def _date(value, where):
    if value is None:
        return None
    if isinstance(value, datetime):
        value = value.date()
    if isinstance(value, date):
        return value.isoformat()
    try:
        return date.fromisoformat(str(value)).isoformat()
    except ValueError as exc:
        raise ValueError(f"{where} must be YYYY-MM-DD or null.") from exc


def _table(value, where):
    value = _text(value, where)
    if len(value.split(".")) != 3 or not all(re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", x) for x in value.split(".")):
        raise ValueError(f"{where} must be a three-part catalog.schema.table name.")
    return value


def validate_similarity_settings(settings):
    allowed = {"raw_weights", "distance_method", "parent_game_process", "neighbor_limits",
               "neighbor_released_after", "minimum_age_gap_months"}
    _keys(settings, allowed, "similarity settings", allowed)
    settings = copy.deepcopy(settings)
    settings["distance_method"] = _text(settings["distance_method"], "distance_method").lower()
    if settings["distance_method"] not in {"cosine", "euclidean"}:
        raise ValueError("distance_method must be cosine or euclidean.")
    if settings["parent_game_process"] not in PARENT_PROCESSES:
        raise ValueError(f"Unknown parent_game_process: {settings['parent_game_process']!r}")
    _keys(settings["raw_weights"], WEIGHT_GROUPS, "raw_weights", WEIGHT_GROUPS)
    for key, value in settings["raw_weights"].items():
        if isinstance(value, bool) or not isinstance(value, Real) or not math.isfinite(float(value)) or value < 0:
            raise ValueError(f"raw_weights.{key} must be a finite nonnegative number.")
        settings["raw_weights"][key] = float(value)
    if not any(settings["raw_weights"].values()):
        raise ValueError("At least one feature-group weight must be positive.")
    _keys(settings["neighbor_limits"], {"sales", "rental", "eilers"}, "neighbor_limits", {"sales", "rental", "eilers"})
    settings["neighbor_limits"] = {k: _integer(v, f"neighbor_limits.{k}") for k, v in settings["neighbor_limits"].items()}
    settings["minimum_age_gap_months"] = _integer(settings["minimum_age_gap_months"], "minimum_age_gap_months")
    settings["neighbor_released_after"] = _date(settings["neighbor_released_after"], "neighbor_released_after")
    if settings["neighbor_released_after"] is None:
        raise ValueError("neighbor_released_after must be a date.")
    return settings


def named_forecast_settings(item, similarity):
    """Weights are ordered by historical demand in each month, highest first."""
    name = item["name"]
    spec = item.get("forecast", {})
    _keys(spec, {"rank_weights", "train_bias_correction"}, f"{name}.forecast")
    weights = spec.get("rank_weights", [1.0])
    if not isinstance(weights, list) or not weights:
        raise ValueError(f"{name}.forecast.rank_weights must be a nonempty list.")
    if any(isinstance(w, bool) or not isinstance(w, Real) or not math.isfinite(w) or w <= 0 for w in weights):
        raise ValueError(f"{name}.forecast.rank_weights must contain positive finite numbers.")
    if any(a < b for a, b in zip(weights, weights[1:])):
        raise ValueError(f"{name}.forecast.rank_weights must be ordered highest weight first.")
    if not math.isfinite(sum(weights)):
        raise ValueError(f"{name}.forecast.rank_weights sum must be finite.")
    correct = spec.get("train_bias_correction", False)
    if not isinstance(correct, bool):
        raise ValueError(f"{name}.forecast.train_bias_correction must be true or false.")
    return {"name": name, "rank_weights": [float(w) for w in weights],
            "train_bias_correction": correct, "similarity_settings_hash": config_sha256(similarity)}


def load_experiment_config(config_path, require_enabled=True):
    try:
        import yaml
    except ImportError as exc:
        raise ImportError("PyYAML is required. Install PyYAML on your Databricks environment, then rerun.") from exc

    # Reject duplicate YAML keys rather than silently replacing a weight/setting.
    class UniqueLoader(yaml.SafeLoader):
        pass
    def unique_mapping(loader, node, deep=False):
        result = {}
        for key_node, value_node in node.value:
            key = loader.construct_object(key_node, deep=deep)
            if key in result:
                raise ValueError(f"Duplicate YAML key {key!r} on line {key_node.start_mark.line + 1}.")
            result[key] = loader.construct_object(value_node, deep=deep)
        return result
    UniqueLoader.add_constructor(yaml.resolver.BaseResolver.DEFAULT_MAPPING_TAG, unique_mapping)
    path = Path(config_path).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f"Configuration not found: {path}. Put experiments.yaml in configs beside this notebook, or set CONFIG_PATH to its absolute /Workspace/... path.")
    original_text = path.read_text(encoding="utf-8")
    raw = yaml.load(original_text, Loader=UniqueLoader)
    sections = {"schema_version", "data", "outputs", "tracking", "monitoring", "similarity_defaults", "experiments"}
    # Downstream stages share this file without changing 05's semantic inputs.
    _keys(raw, sections | {"forecast", "evaluation"}, "configuration", sections)
    if isinstance(raw["schema_version"], bool) or raw["schema_version"] != 1:
        raise ValueError("Only schema_version: 1 is supported.")
    _keys(raw["data"], {"revision", "ai_attributes_table", "source_version", "prediction_as_of_date"}, "data",
          {"revision", "ai_attributes_table", "source_version", "prediction_as_of_date"})
    data = copy.deepcopy(raw["data"])
    data["revision"] = _text(data["revision"], "data.revision")
    data["ai_attributes_table"] = _table(data["ai_attributes_table"], "data.ai_attributes_table")
    if data["source_version"] is not None:
        data["source_version"] = _integer(data["source_version"], "data.source_version")
    data["prediction_as_of_date"] = _date(data["prediction_as_of_date"], "data.prediction_as_of_date")
    output_keys = {"like_games_table", "runs_table", "attempts_table"}
    _keys(raw["outputs"], output_keys, "outputs", output_keys)
    outputs = {k: _table(v, f"outputs.{k}") for k, v in raw["outputs"].items()}
    if len(set(outputs.values())) != len(outputs) or data["ai_attributes_table"] in outputs.values():
        raise ValueError("Output tables must be distinct and cannot be the AI source table.")
    _keys(raw["tracking"], {"experiment_path"}, "tracking", {"experiment_path"})
    tracking_path = _text(raw["tracking"]["experiment_path"], "tracking.experiment_path")
    if not tracking_path.startswith("/"):
        raise ValueError("tracking.experiment_path must start with /.")
    _keys(raw["monitoring"], {"history_rows"}, "monitoring", {"history_rows"})
    history_rows = _integer(raw["monitoring"]["history_rows"], "monitoring.history_rows", 1)
    defaults = validate_similarity_settings(raw["similarity_defaults"])
    if not isinstance(raw["experiments"], list) or not raw["experiments"]:
        raise ValueError("experiments must be a nonempty list.")
    experiments, names = [], set()
    for index, item in enumerate(raw["experiments"]):
        _keys(item, {"name", "enabled", "overrides", "forecast"}, f"experiments[{index}]", {"name"})
        name = _text(item["name"], f"experiments[{index}].name")
        if name in names:
            raise ValueError(f"Duplicate experiment name: {name!r}")
        names.add(name)
        enabled = item.get("enabled", True)
        if not isinstance(enabled, bool):
            raise ValueError(f"{name}.enabled must be true or false.")
        overrides = item.get("overrides", {})
        _keys(overrides, defaults, f"{name}.overrides")
        merged = copy.deepcopy(defaults)
        for key, value in overrides.items():
            if key in {"raw_weights", "neighbor_limits"}:
                _keys(value, defaults[key], f"{name}.overrides.{key}")
                merged[key].update(copy.deepcopy(value))
            else:
                merged[key] = copy.deepcopy(value)
        merged = validate_similarity_settings(merged)
        forecast = named_forecast_settings(item, merged)
        experiments.append({"name": name, "enabled": enabled, "settings": merged, "forecast": forecast})
    if require_enabled and not any(x["enabled"] for x in experiments):
        raise ValueError("Enable at least one experiment in the YAML file.")
    return {"data": data, "outputs": outputs, "tracking": {"experiment_path": tracking_path},
            "monitoring": {"history_rows": history_rows}, "experiments": experiments,
            "config_path": str(path), "config_text": original_text,
            "yaml_sha256": hashlib.sha256(original_text.encode("utf-8")).hexdigest()}


def similarity_work_identity(experiment, data, source_version):
    semantic = {"implementation_version": SIMILARITY_IMPLEMENTATION_VERSION,
                "data_revision": data["revision"], "source_table": data["ai_attributes_table"],
                "source_table_version": int(source_version),
                "prediction_as_of_date": data["prediction_as_of_date"],
                "similarity": copy.deepcopy(experiment["settings"])}
    digest = config_sha256(semantic)
    return digest, semantic


def preflight_similarity_tables(spark, config):
    """Refuse to mix pre-ledger legacy generations with the new history."""
    results, runs = (config["outputs"][x] for x in ["like_games_table", "runs_table"])
    if spark.catalog.tableExists(results):
        frame = spark.table(results)
        if frame.limit(1).count():
            if "similarity_work_key" not in frame.columns or not spark.catalog.tableExists(runs):
                raise ValueError("Previous similarity results use the old format. Run 00_Reset_Similarity_Pipeline once before the first new batch. No results were overwritten.")
            ledger = spark.table(runs)
            if not {"work_key", "experiment_id", "metadata_json"}.issubset(ledger.columns):
                raise ValueError("The similarity status table has an incompatible schema. Run the one-time reset before starting.")
            if frame.select("experiment_id").distinct().join(ledger.select("experiment_id").distinct(), "experiment_id", "left_anti").limit(1).count():
                raise ValueError("Similarity rows without a matching status record were found. Resolve/reset them before running a new batch.")


def resolve_similarity_source_version(spark, config):
    """A data revision pins one source snapshot across later manually run batches."""
    from pyspark.sql import functions as F
    data, runs = config["data"], config["outputs"]["runs_table"]
    pinned = []
    if spark.catalog.tableExists(runs):
        ledger = spark.table(runs)
        if "metadata_json" not in ledger.columns:
            raise ValueError("Incompatible similarity status table; perform the one-time reset first.")
        pinned = [int(r["source_table_version"]) for r in ledger.where(
            (F.get_json_object("metadata_json", "$.data_revision") == data["revision"])
            & (F.col("source_path") == data["ai_attributes_table"])
        ).select("source_table_version").distinct().collect()]
    if len(set(pinned)) > 1:
        raise ValueError("This data revision has multiple source snapshots. Choose a new revision after resolving the history.")
    if pinned:
        version = pinned[0]
        if data["source_version"] is not None and data["source_version"] != version:
            raise ValueError(f"Revision {data['revision']!r} is already pinned to Delta version {version}. Change data.revision to use another version.")
        return version
    if data["source_version"] is not None:
        return data["source_version"]
    return int(spark.sql(f"DESCRIBE HISTORY {quoted_table(data['ai_attributes_table'])}").agg(F.max("version").alias("version")).first()["version"])


In [0]:
"""Notebook 06 YAML validation, immutable work keys, and source-revision pinning."""
import copy
import hashlib
import json
import math
from numbers import Integral, Real

FORECAST_IMPLEMENTATION_VERSION = "06_batch_yaml_v1_2026_11_19"
FORECAST_SOURCE_KEYS = {"sales", "lease", "performance"}
FORECAST_OUTPUT_KEYS = {"demand", "performance", "neighbors", "runs", "attempts"}
DEFAULT_HISTORY_MONTHS = {"sales_min_observed_months": 6, "rental_min_observed_months": 9}
FORECAST_SETTING_KEYS = {"distance_cutoff", "minimum_neighbors", "horizon_months", "slot_counts",
                         "lease_value_column", "lease_metric_name", "lease_curve_label", "performance_metrics"}


def load_forecast_config(config_path):
    # Shared loader rejects duplicate YAML keys and validates the existing 05
    # sections. 05 ignores this downstream section for work identity purposes.
    common = load_experiment_config(config_path, require_enabled=False)
    import yaml
    raw = yaml.safe_load(common["config_text"])
    forecast = raw.get("forecast")
    forecast_keys = {"data_revision", "sources", "source_versions", "settings", "outputs", "tracking", "selection", "monitoring"}
    if isinstance(forecast, dict) and "experiments" in forecast:
        raise ValueError("Move forecast.experiments into the named entries in the top-level experiments list.")
    _keys(forecast, forecast_keys, "forecast", forecast_keys)
    revision = _text(forecast["data_revision"], "forecast.data_revision")
    _keys(forecast["sources"], FORECAST_SOURCE_KEYS, "forecast.sources", FORECAST_SOURCE_KEYS)
    sources = {k: _table(v, f"forecast.sources.{k}") for k, v in forecast["sources"].items()}
    _keys(forecast["source_versions"], FORECAST_SOURCE_KEYS, "forecast.source_versions", FORECAST_SOURCE_KEYS)
    versions = {k: None if v is None else _integer(v, f"forecast.source_versions.{k}")
                for k, v in forecast["source_versions"].items()}
    _keys(forecast["outputs"], FORECAST_OUTPUT_KEYS, "forecast.outputs", FORECAST_OUTPUT_KEYS)
    outputs = {k: _table(v, f"forecast.outputs.{k}") for k, v in forecast["outputs"].items()}
    protected_inputs = {common["data"]["ai_attributes_table"], *sources.values(), *common["outputs"].values()}
    if len(set(outputs.values())) != len(outputs) or set(outputs.values()) & protected_inputs:
        raise ValueError("Forecast output tables must be distinct and cannot replace any source or similarity tables.")
    _keys(forecast["settings"], FORECAST_SETTING_KEYS | set(DEFAULT_HISTORY_MONTHS), "forecast.settings", FORECAST_SETTING_KEYS)
    settings = copy.deepcopy(forecast["settings"])
    for key, default in DEFAULT_HISTORY_MONTHS.items():
        settings[key] = _integer(settings.get(key, default), key, 1)
    cutoff = settings["distance_cutoff"]
    if isinstance(cutoff, bool) or not isinstance(cutoff, Real) or not math.isfinite(float(cutoff)) or cutoff < 0:
        raise ValueError("forecast.settings.distance_cutoff must be finite and >= 0.")
    settings["distance_cutoff"] = float(cutoff)
    settings["minimum_neighbors"] = _integer(settings["minimum_neighbors"], "minimum_neighbors")
    settings["horizon_months"] = _integer(settings["horizon_months"], "horizon_months", 1)
    slots = settings["slot_counts"]
    if not isinstance(slots, list) or any(isinstance(x, bool) or not isinstance(x, Real) or not math.isfinite(float(x)) or x <= 0 for x in slots):
        raise ValueError("slot_counts must be a list of finite positive numbers; [] means all positive counts.")
    if len(set(slots)) != len(slots):
        raise ValueError("slot_counts cannot contain duplicates.")
    settings["slot_counts"] = sorted(float(x) for x in slots)
    for key in ["lease_value_column", "lease_metric_name", "lease_curve_label"]:
        settings[key] = _text(settings[key], key)
    if not isinstance(settings["performance_metrics"], list) or not settings["performance_metrics"]:
        raise ValueError("performance_metrics must be a nonempty list.")
    settings["performance_metrics"] = [_text(x, "performance_metrics entry") for x in settings["performance_metrics"]]
    if len(set(settings["performance_metrics"])) != len(settings["performance_metrics"]):
        raise ValueError("performance_metrics cannot contain duplicates.")
    settings["performance_metrics"] = sorted(settings["performance_metrics"])
    _keys(forecast["tracking"], {"experiment_path"}, "forecast.tracking", {"experiment_path"})
    mlflow_path = _text(forecast["tracking"]["experiment_path"], "forecast.tracking.experiment_path")
    if not mlflow_path.startswith("/"):
        raise ValueError("forecast.tracking.experiment_path must be an absolute workspace path.")
    _keys(forecast["selection"], {"similarity_experiment_ids"}, "forecast.selection", {"similarity_experiment_ids"})
    ids = forecast["selection"]["similarity_experiment_ids"]
    if not isinstance(ids, list):
        raise ValueError("similarity_experiment_ids must be a list; [] discovers all completed 05 experiments.")
    ids = [_text(x, "similarity_experiment_ids entry") for x in ids]
    if len(set(ids)) != len(ids):
        raise ValueError("similarity_experiment_ids cannot contain duplicates.")
    _keys(forecast["monitoring"], {"history_rows"}, "forecast.monitoring", {"history_rows"})
    return {"common": common,
            "data_revision": revision, "sources": sources, "source_versions": versions,
            "settings": settings, "outputs": outputs, "tracking": {"experiment_path": mlflow_path},
            "selection": {"similarity_experiment_ids": ids},
            "monitoring": {"history_rows": _integer(forecast["monitoring"]["history_rows"], "forecast.monitoring.history_rows", 1)}}


def forecast_work_identity(upstream, config, source_versions):
    # Descriptive labels, requested batch membership, MLflow IDs/paths, output
    # locations, and the global appended like-games table version are not inputs.
    settings = {k: copy.deepcopy(v) for k, v in config["settings"].items() if k != "lease_curve_label"}
    semantic = {
        "implementation_version": FORECAST_IMPLEMENTATION_VERSION,
        "forecast_data_revision": config["data_revision"],
        "similarity_experiment_id": upstream["experiment_id"],
        "similarity_work_key": upstream["work_key"],
        "similarity_config_hash": upstream["config_hash"],
        "similarity_result_hash": upstream["result_hash"],
        "source_paths": copy.deepcopy(config["sources"]),
        "source_versions_resolved": {k: int(source_versions[k]) for k in sorted(FORECAST_SOURCE_KEYS)},
        **settings,
        "prediction_date_rule": "saved similarity prediction_date, falling back to target game_release_date",
        "neighbor_observation_rule": "source_month < first_day_of_target_prediction_month",
        "reporting_assumption": "completed prior month available; publication timestamps not supplied",
        "calendar_rule": "target forecast starts at explicit prediction month; neighbor age starts at first historically observed source month",
        "target_actual_policy": "predictions only; actuals loaded independently by versioned evaluation",
        "sales_fill": "internal missing monthly rows zero, then cumulative sum; no trailing fill",
        "lease_fill": "internal missing monthly levels forward-filled; no trailing fill",
        "performance_fill": "internal missing per-metric neighbor statistics forward-filled; leading null remains; no trailing fill",
        "performance_weighting": "positive no_of_slots; per-metric nonnull denominator",
        "neighbor_selection": "saved channel list; deduplicate source names, exclude target mapping, cutoff then nearest minimum fallback",
        "p3_parent_after_cutoff": "normal cutoff/fallback applies to saved parent candidate",
        "bands": "population standard deviation across contributing neighbors; mean +/- 2 std, not calibrated intervals",
    }
    return config_sha256(semantic), semantic


def preflight_forecast_tables(spark, config):
    outputs = config["outputs"]
    for label in ["demand", "performance", "neighbors"]:
        path = outputs[label]
        if spark.catalog.tableExists(path):
            frame = spark.table(path)
            if frame.limit(1).count():
                if "forecast_work_key" not in frame.columns or not spark.catalog.tableExists(outputs["runs"]):
                    raise ValueError("Legacy forecast results were found. Clear only the old 06/07 outputs using reset scope '06_07' before this migration. Preserve your new 05 results.")
                runs = spark.table(outputs["runs"])
                if not {"work_key", "forecast_run_id", "metadata_json"}.issubset(runs.columns):
                    raise ValueError("The forecast status table uses the old schema. Migrate/reset scope '06_07'; preserve 05.")
                if frame.select("forecast_run_id").distinct().join(runs.select("forecast_run_id").distinct(), "forecast_run_id", "left_anti").limit(1).count():
                    raise ValueError("Forecast rows without status records were found. Resolve them before retrying this batch.")


def resolve_forecast_source_versions(spark, config):
    from pyspark.sql import functions as F
    table = config["outputs"]["runs"]
    matches = []
    if spark.catalog.tableExists(table):
        ledger = spark.table(table)
        if "metadata_json" not in ledger.columns:
            raise ValueError("Legacy forecast manifest found. Reset scope '06_07' before using new 06; preserve 05.")
        matches = ledger.where(F.get_json_object("metadata_json", "$.data_revision") == config["data_revision"]).select(
            "source_versions_json", "config_json").distinct().collect()
    versions = None
    for row in matches:
        previous_config = json.loads(row["config_json"])
        if previous_config["source_paths"] != config["sources"]:
            raise ValueError("This forecast data revision is already assigned to other source tables. Choose another forecast.data_revision.")
        candidate = {k: int(v) for k, v in json.loads(row["source_versions_json"]).items()}
        if versions is not None and candidate != versions:
            raise ValueError("This forecast revision contains conflicting source snapshots; resolve its history before running.")
        versions = candidate
    if versions is not None:
        for key, requested in config["source_versions"].items():
            if requested is not None and requested != versions[key]:
                raise ValueError(f"Revision is pinned to {key} version {versions[key]}; change forecast.data_revision to refresh.")
        return versions
    return {key: int(requested) if requested is not None else int(
        spark.sql(f"DESCRIBE HISTORY {quoted_table(config['sources'][key])}").agg(F.max("version").alias("v")).first()["v"])
        for key, requested in config["source_versions"].items()}


In [0]:
"""07 configuration, fixed-label source revisions, and semantic identities."""
import copy
import hashlib
import json
import math
from numbers import Real

EVALUATION_IMPLEMENTATION_VERSION = "07_demand_curve_metrics_v2"
# Saved 06 demand columns. Eilers has one index forecast per metric.
DEMAND_CURVES = {
    "baseline": ("forecast_value", "Neighbor mean"),
    "linear": ("forecast_linear", "Linear"),
    "poly2": ("forecast_poly2", "Quadratic"),
    "log": ("forecast_log", "Logarithmic"),
}
EVALUATION_SOURCE_KEYS = {"sales", "lease", "performance", "mappings"}
EVALUATION_OUTPUT_KEYS = {"cohorts", "cohort_runs", "splits", "eligibility", "metrics", "selected",
                          "selection_runs", "runs", "attempts", "actuals_snapshots", "batches"}
EVALUATION_SETTING_KEYS = {"evaluation_months", "minimum_observed_months", "bracket_month",
                           "validation_fraction", "test_fraction", "exclude_pre_release_curves",
                           "exclude_cumulative_imputed_history", "performance_metrics"}


def load_evaluation_config(config_path):
    forecast = load_forecast_config(config_path)
    common = forecast["common"]
    import yaml
    section = yaml.safe_load(common["config_text"]).get("evaluation")
    keys = {"data_revision", "sources", "source_versions", "settings", "outputs", "tracking", "selection", "monitoring"}
    _keys(section, keys, "evaluation", keys)
    _keys(section["sources"], EVALUATION_SOURCE_KEYS, "evaluation.sources", EVALUATION_SOURCE_KEYS)
    sources = {k: _table(v, f"evaluation.sources.{k}") for k, v in section["sources"].items()}
    _keys(section["source_versions"], EVALUATION_SOURCE_KEYS, "evaluation.source_versions", EVALUATION_SOURCE_KEYS)
    versions = {k: None if v is None else _integer(v, f"evaluation.source_versions.{k}")
                for k, v in section["source_versions"].items()}
    _keys(section["outputs"], EVALUATION_OUTPUT_KEYS, "evaluation.outputs", EVALUATION_OUTPUT_KEYS)
    outputs = {k: _table(v, f"evaluation.outputs.{k}") for k, v in section["outputs"].items()}
    protected = {*common["outputs"].values(), common["data"]["ai_attributes_table"],
                 *forecast["outputs"].values(), *forecast["sources"].values(), *sources.values()}
    if len(set(outputs.values())) != len(outputs) or set(outputs.values()) & protected:
        raise ValueError("Evaluation outputs must be distinct and cannot replace source, 05, or 06 tables.")
    _keys(section["settings"], EVALUATION_SETTING_KEYS, "evaluation.settings", EVALUATION_SETTING_KEYS)
    settings = copy.deepcopy(section["settings"])
    for key in ["evaluation_months", "minimum_observed_months", "bracket_month"]:
        settings[key] = _integer(settings[key], f"evaluation.settings.{key}", 1)
    if settings["bracket_month"] > settings["evaluation_months"] or settings["minimum_observed_months"] > settings["evaluation_months"]:
        raise ValueError("Minimum observed months and bracket month cannot exceed the evaluation horizon.")
    for key in ["validation_fraction", "test_fraction"]:
        value = settings[key]
        if isinstance(value, bool) or not isinstance(value, Real) or not math.isfinite(float(value)) or not 0 < value < 1:
            raise ValueError(f"{key} must be finite and strictly between 0 and 1.")
        settings[key] = float(value)
    if settings["validation_fraction"] + settings["test_fraction"] >= 1:
        raise ValueError("validation_fraction + test_fraction must be < 1.")
    for key in ["exclude_pre_release_curves", "exclude_cumulative_imputed_history"]:
        if not isinstance(settings[key], bool):
            raise ValueError(f"{key} must be true or false.")
    metrics = settings["performance_metrics"]
    if not isinstance(metrics, list) or not metrics:
        raise ValueError("evaluation performance_metrics must be a nonempty list.")
    metrics = [_text(m, "evaluation performance metric") for m in metrics]
    if len(metrics) != len(set(metrics)):
        raise ValueError("Duplicate evaluation performance metrics.")
    settings["performance_metrics"] = sorted(metrics)
    _keys(section["tracking"], {"experiment_path"}, "evaluation.tracking", {"experiment_path"})
    tracking = _text(section["tracking"]["experiment_path"], "evaluation.tracking.experiment_path")
    if not tracking.startswith("/"):
        raise ValueError("evaluation tracking path must be an absolute workspace path.")
    _keys(section["selection"], {"forecast_run_ids"}, "evaluation.selection", {"forecast_run_ids"})
    ids = section["selection"]["forecast_run_ids"]
    if not isinstance(ids, list):
        raise ValueError("evaluation.selection.forecast_run_ids must be a list.")
    ids = [_text(x, "forecast_run_id") for x in ids]
    if len(ids) != len(set(ids)):
        raise ValueError("Duplicate evaluation forecast IDs.")
    _keys(section["monitoring"], {"history_rows"}, "evaluation.monitoring", {"history_rows"})
    return {"common": common, "forecast": forecast,
            "data_revision": _text(section["data_revision"], "evaluation.data_revision"),
            "sources": sources, "source_versions": versions, "settings": settings, "outputs": outputs,
            "tracking": {"experiment_path": tracking}, "selection": {"forecast_run_ids": ids},
            "monitoring": {"history_rows": _integer(section["monitoring"]["history_rows"], "history_rows", 1)}}


def evaluation_policy(config):
    return {"demand_bias_policy": "monthly training residuals when enabled in the saved named forecast",
            **copy.deepcopy(config["settings"]),
            "missing_release_prediction_date": config["common"]["data"]["prediction_as_of_date"],
            "target_registry": "independent pinned AI mapping/attributes snapshot, not forecast availability",
            "actual_calendar": "source calendar months aligned to target prediction month",
            "sales_actuals": "monthly observed sales summed then cumulative; internal missing months flagged",
            "rental_actuals": "observed monthly source quantity; no target forward fill",
            "performance_actuals": "duplicate slot buckets averaged then positive-slot metric-specific weighting; no target fill",
            "split_rule": "canonical game chronological split; equal release dates stay together; freeze with cohort",
            "selection_rule": "validation MAE, RMSE, forecast run ID and curve name; full coverage required",
            "implementation_version": EVALUATION_IMPLEMENTATION_VERSION}


def evaluation_family(configuration, config):
    """Separate incompatible business populations; source vintages may compare."""
    return {"source_paths": {key: configuration["source_paths"][key] for key in ["sales", "lease", "performance"]},
            "slot_counts": sorted(float(x) for x in configuration.get("slot_counts", [])),
            "lease_value_column": configuration.get("lease_value_column", "source_row_count"),
            "lease_metric_name": configuration.get("lease_metric_name", "monthly_rental_records"),
            "performance_metrics": list(config["settings"]["performance_metrics"]),
            "prediction_date_rule": configuration.get("prediction_date_rule"),
            "calendar_rule": configuration.get("calendar_rule"),
            "neighbor_observation_rule": configuration.get("neighbor_observation_rule")}


def evaluation_work_identity(forecast_record, cohort_id, policy_hash):
    semantic = {"implementation_version": EVALUATION_IMPLEMENTATION_VERSION,
                "forecast_run_id": forecast_record["forecast_run_id"],
                "forecast_config_hash": forecast_record["forecast_config_hash"],
                "cohort_id": cohort_id, "policy_hash": policy_hash}
    return config_sha256(semantic), semantic


SNAPSHOT_SCHEMA = "snapshot_id string, data_revision string, source_paths_json string, source_versions_json string, created_at_utc string"
BATCH_SCHEMA = "batch_id string, data_revision string, status string, started_at_utc string, updated_at_utc string, summary_json string, errors_json string"


def evaluation_merge_record(spark, table, schema, key, record):
    import uuid
    name = "evaluation_record_" + uuid.uuid4().hex
    spark.createDataFrame([record], schema=schema).createOrReplaceTempView(name)
    try:
        spark.sql(f"MERGE INTO {quoted_table(table)} target USING `{name}` source ON target.`{key}` = source.`{key}` "
                  "WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *")
    finally:
        spark.catalog.dropTempView(name)


def resolve_evaluation_source_versions(spark, config):
    """Pin actuals/mappings before cohort preparation; no moving latest labels."""
    from datetime import datetime, timezone
    from pyspark.sql import functions as F
    path = config["outputs"]["actuals_snapshots"]
    spark.sql(f"CREATE TABLE IF NOT EXISTS {quoted_table(path)} ({SNAPSHOT_SCHEMA}) USING DELTA")
    snapshot_id = config_sha256({"revision": config["data_revision"], "sources": config["sources"]})
    existing = spark.table(path).where(F.col("data_revision") == config["data_revision"]).limit(2).collect()
    if len(existing) > 1:
        raise ValueError("Multiple actuals snapshots own this label revision; resolve before evaluation.")
    if existing:
        record = existing[0].asDict()
        if json.loads(record["source_paths_json"]) != config["sources"]:
            raise ValueError("This evaluation revision is assigned to different source tables; change evaluation.data_revision.")
        versions = {k: int(v) for k, v in json.loads(record["source_versions_json"]).items()}
        if any(v is not None and v != versions[k] for k, v in config["source_versions"].items()):
            raise ValueError("The label revision is already pinned. Change evaluation.data_revision to refresh source versions.")
        return versions
    versions = {key: int(requested) if requested is not None else int(
        spark.sql(f"DESCRIBE HISTORY {quoted_table(config['sources'][key])}").agg(F.max("version").alias("v")).first()["v"])
        for key, requested in config["source_versions"].items()}
    evaluation_merge_record(spark, path, SNAPSHOT_SCHEMA, "snapshot_id", {
        "snapshot_id": snapshot_id, "data_revision": config["data_revision"],
        "source_paths_json": config_json(config["sources"]), "source_versions_json": config_json(versions),
        "created_at_utc": datetime.now(timezone.utc).isoformat()})
    return versions


def preflight_evaluation_tables(spark, config):
    outputs = config["outputs"]
    for label, required in [("runs", {"work_key", "evaluation_id", "metadata_json"}),
                            ("cohorts", {"cohort_id", "target_game_key"}),
                            ("metrics", {"evaluation_id", "policy_hash"})]:
        path = outputs[label]
        if spark.catalog.tableExists(path):
            frame = spark.table(path)
            if frame.limit(1).count() and not required.issubset(frame.columns):
                raise ValueError("Legacy 07 outputs were found. Use reset scope '07' once to preserve new 05/06 results.")


BASE_DEMAND_CURVES = dict(DEMAND_CURVES)
for base, (_, label) in BASE_DEMAND_CURVES.items():
    DEMAND_CURVES[base + "_train_bias"] = ("adjusted_" + base, label + " (bias-corrected)")


def apply_training_bias(frame, calibration, curve, raw_prediction, origin_column="prediction_date"):
    """Apply the saved additive offset only to forecasts issued after training closed."""
    import numpy as np
    import pandas as pd
    result = pd.Series(np.nan, index=frame.index, dtype=float)
    cutoff = calibration.get("available_from")
    records = calibration.get("offsets", {}).get(curve, [])
    if not cutoff or not records or frame.empty:
        return result
    offsets = pd.DataFrame(records).set_index(["revenue_type", "month_index"])["offset"]
    keys = pd.MultiIndex.from_frame(frame[["revenue_type", "month_index"]])
    adjusted = pd.to_numeric(raw_prediction, errors="coerce") + offsets.reindex(keys).to_numpy()
    available = pd.to_datetime(frame[origin_column], errors="coerce").ge(pd.Timestamp(cutoff))
    return adjusted.clip(lower=0).where(available & np.isfinite(adjusted))


def with_curve_name(frame):
    """Earlier 07 evaluations contain only the raw 06 baseline."""
    from pyspark.sql import functions as F
    curve = F.coalesce("curve_name", F.lit("baseline")) if "curve_name" in frame.columns else F.lit("baseline")
    return frame.withColumn("curve_name", curve)


In [0]:
"""Read historical observations for the like games retained by the selected 06 run."""

def review_name(value):
    import pandas as pd

    if pd.isna(value):
        return None
    key = " ".join(str(value).lower().split())
    return None if key in {"", "none", "null", "nan", "na", "n/a", "<na>"} else key


def build_neighbor_roster(audit, mappings, channel):
    """Use each list's saved match first, then the game's own mapping for the other source."""
    import pandas as pd

    commercial_type = "sales" if channel == "sales" else "rental"
    games = {}
    for row in audit.itertuples(index=False):
        kind = row.neighbor_type
        if kind not in {commercial_type, "eilers"}:
            continue
        game = games.setdefault(row.neighbor_game, {
            "Like game": row.neighbor_game, "commercial_rank": None, "eilers_rank": None,
            "commercial_key": None, "eilers_key": None,
            "Distance": row.distance, "Similarity": row.similarity_score,
        })
        role = "eilers" if kind == "eilers" else "commercial"
        game[role + "_rank"] = int(row.neighbor_rank)
        game[role + "_key"] = review_name(row.matched_source_name)

    for game in games.values():
        mapping = mappings.get(review_name(game["Like game"]), {})
        for role in ("commercial", "eilers"):
            if game[role + "_key"] is None:
                game[role + "_key"] = review_name(mapping.get(role + "_key"))
    roster = pd.DataFrame(games.values())
    if not roster.empty:
        for col in ("commercial_rank", "eilers_rank"):
            roster[col] = roster[col].astype("Int64")
        roster = roster.sort_values(["commercial_rank", "eilers_rank", "Like game"]).reset_index(drop=True)
    return roster


def load_neighbor_months(spark, config, source, source_keys, cutoff_month, channel):
    """Read one pinned Delta source; collect at most the saved horizon per matched key."""
    import pandas as pd
    from pyspark.sql import functions as F
    from pyspark.sql.window import Window

    if not source_keys:
        return pd.DataFrame()
    path = config["source_paths"][source]
    version = int(config["source_versions_resolved"][source])
    raw = spark.read.option("versionAsOf", version).table(path)

    def norm(name):
        return F.regexp_replace(F.lower(F.trim(F.col(name).cast("string"))), r"\s+", " ")

    def finite(name):
        value = F.col(name).cast("double")
        return F.when(value.isNotNull() & ~F.isnan(value) & (F.abs(value) != float("inf")), value)

    if source == "performance":
        key, date = "game_name", F.to_date(F.col("yearmonth").cast("string"), "yyyy-MM-dd")
    elif source == "sales":
        key, date = "normalized_game_name", F.to_date("beginning_month_date")
    else:
        key, date = "theme_name_eilers", F.to_date("month")
    frame = (raw.withColumn("source_key", norm(key)).withColumn("source_month", F.trunc(date, "month"))
             .where(F.col("source_key").isin(source_keys) & (F.col("source_month") < F.lit(cutoff_month))))
    group = ["source_key", "source_month"]
    if source == "performance":
        metrics = config["performance_metrics"]
        slots = config["slot_counts"]
        status = "owned" if channel == "sales" else "leased"
        frame = frame.where(norm("own_status") == status).withColumn("slots", finite("no_of_slots")).where(F.col("slots") > 0)
        if slots:
            frame = frame.where(F.col("slots").isin(slots))
        buckets = frame.groupBy(*group, "slots").agg(*[F.avg(finite(m)).alias(m) for m in metrics])
        weighted = []
        for metric in metrics:
            weight = F.when(F.col(metric).isNotNull(), F.col("slots"))
            total = F.sum(weight)
            weighted.append(F.when(total > 0, F.sum(weight * F.col(metric)) / total).alias(metric))
        monthly = buckets.groupBy(*group).agg(*weighted)
    else:
        value_col = "expr1_sum" if source == "sales" else config["lease_value_column"]
        if source == "sales":
            frame = frame.where(norm("revenue_type").isin("sale", "sales"))
        monthly = (frame.withColumn("monthly_value", finite(value_col)).where(F.col("monthly_value").isNotNull())
                   .groupBy(*group).agg(F.sum("monthly_value").alias("monthly_value")))

    by_game = Window.partitionBy("source_key")
    monthly = (monthly.withColumn("first_month", F.min("source_month").over(by_game))
               .withColumn("month_index", F.months_between("source_month", "first_month").cast("int") + 1)
               .where(F.col("month_index") <= int(config["horizon_months"])).drop("first_month"))
    if source == "sales":
        monthly = monthly.withColumn("cumulative_sales", F.sum("monthly_value").over(by_game.orderBy("month_index")))
    return monthly.orderBy("source_key", "month_index").toPandas()


def show_neighbor_review(spark, audit_frame, demand_table, run_meta, channel, prediction_date, curve_label):
    import json
    import matplotlib.pyplot as plt
    import pandas as pd
    from IPython.display import HTML, display
    from pyspark.sql import functions as F

    display(HTML("<h3>Like games and historical results</h3>"))
    if audit_frame is None:
        print("The saved neighbor table is unavailable for this run.")
        return
    audit = audit_frame.select("neighbor_type", "neighbor_rank", "neighbor_game", "matched_source_name",
                               "distance", "similarity_score", "has_asof_source_data").toPandas()
    if audit.empty:
        print("No retained like games for this target and channel.")
        return

    game_keys = sorted(audit["neighbor_game"].map(review_name).dropna().unique().tolist())
    match_column = "to_sales_name_match" if channel == "sales" else "to_rental_name_match"
    mapping_rows = (spark.table(demand_table)
                    .where((F.col("forecast_run_id") == run_meta["forecast_run_id"]) &
                           (F.col("revenue_type") == channel) & (F.col("month_index") == 1) &
                           F.col("target_game_key").isin(game_keys))
                    .select("target_game_key", F.col(match_column).alias("commercial_key"),
                            F.col("to_eilers_ts_name_match").alias("eilers_key")).collect())
    mappings = {r["target_game_key"]: r.asDict() for r in mapping_rows}
    roster = build_neighbor_roster(audit, mappings, channel)
    commercial = "Sale" if channel == "sales" else "Rent"
    names = {"commercial_rank": commercial + " rank", "eilers_rank": "Eilers rank",
             "commercial_key": commercial + " source", "eilers_key": "Eilers source"}

    def table(frame):
        return '<div style="overflow-x:auto">' + frame.to_html(index=False, border=0, na_rep="—", escape=True) + '</div>'

    print("Ranks show which list retained each game. A blank rank means its history is shown for comparison only.")
    summary = roster.rename(columns=names).copy()
    for col in names.values():
        summary[col] = summary[col].astype(object).where(summary[col].notna(), "—")
    display(HTML(table(summary[["Like game", commercial + " rank", "Eilers rank", "Distance", "Similarity",
                                commercial + " source", "Eilers source"]])))
    missing = audit.loc[audit["has_asof_source_data"].eq(False)]
    if not missing.empty:
        print("Retained without prior source data: " + ", ".join(
            f"{row.neighbor_game} ({row.neighbor_type})" for row in missing.itertuples(index=False)))
    if pd.isna(prediction_date):
        print("Monthly histories need a saved target prediction date.")
        return
    cutoff = pd.Timestamp(prediction_date).replace(day=1).date()
    config = json.loads(run_meta.get("config_json") or "{}")
    if not {"source_paths", "source_versions_resolved", "horizon_months", "performance_metrics"}.issubset(config):
        print("The saved run configuration is missing the source snapshots or history settings.")
        return

    horizon = int(config["horizon_months"])
    print(f"Observed source months before {cutoff:%Y-%m-%d}. Month 1 is each game's first month in each source. "
          "Gaps indicate missing observations.")
    panels, details = [], []
    for source, key_column in [(channel, "commercial_key"), ("performance", "eilers_key")]:
        label = "Eilers" if source == "performance" else commercial
        keys = sorted(roster[key_column].dropna().unique().tolist())
        try:
            monthly = load_neighbor_months(spark, config, source, keys, cutoff, channel)
        except Exception as error:
            print(f"{label} history unavailable for the saved snapshot. {type(error).__name__}: {error}")
            continue
        if monthly.empty:
            print(f"No {label} history for these like games in the selected period and channel.")
            continue
        observations = roster[["Like game", key_column]].merge(monthly, left_on=key_column, right_on="source_key")
        if source == "performance":
            fields = config["performance_metrics"]
            labels = [field.replace("avg_", "").replace("_", " ") for field in fields]
            panels.extend((observations, field, title, "Index") for field, title in zip(fields, labels))
        else:
            curve = "cumulative_sales" if channel == "sales" else "monthly_value"
            fields = ["monthly_value", curve] if channel == "sales" else [curve]
            labels = ["Monthly sales", "Cumulative sales"] if channel == "sales" else [curve_label]
            panels.append((observations, curve, curve_label, curve_label))
        observed_games = observations.dropna(subset=fields, how="all")["Like game"]
        missing_games = roster.loc[~roster["Like game"].isin(observed_games), "Like game"]
        if not missing_games.empty:
            print(f"No observed {label} values for: " + ", ".join(missing_games))
        cols = ["Like game", "source_month", "month_index", *fields]
        shown = observations.sort_values(["Like game", "month_index"])[cols].copy()
        shown.columns = ["Like game", "Source month", "Relative month", *labels]
        details.append((label, shown))

    if panels:
        colors = {name: plt.get_cmap("tab20")(index % 20) for index, name in enumerate(roster["Like game"])}
        figure, axes = plt.subplots(len(panels), 1, figsize=(12, 3.0 * len(panels)), squeeze=False, constrained_layout=True)
        for axis, (frame, field, title, ylabel) in zip(axes[:, 0], panels):
            for name, series in frame.groupby("Like game", sort=False):
                values = series.set_index("month_index")[field].reindex(range(1, horizon + 1))
                if values.notna().any():
                    axis.plot(values.index, values, marker=".", color=colors[name], label=name)
            axis.set(title=title, xlabel="Month since first source observation", ylabel=ylabel)
            axis.grid(alpha=0.2)
            if axis.lines:
                axis.legend(loc="upper left", bbox_to_anchor=(1, 1), fontsize=8, frameon=False)
            else:
                axis.text(0.02, 0.85, "No observed values for this metric", transform=axis.transAxes)
        display(figure)
        plt.close(figure)
    for label, rows in details:
        display(HTML(f"<details><summary>{label} monthly values ({len(rows)} game-months)</summary>"
                     + table(rows) + "</details>"))


In [0]:
"""Read-only interactive review of immutable 06 forecasts and saved 07 labels."""


def build_saved_review_options(available_records, eligibility_records, view="test"):
    """Return game/channel choices and permitted runs without mixing Sale/Rent."""
    eligibility = {}
    for item in eligibility_records:
        key = (str(item["forecast_run_id"]), str(item["target_game_key"]), item["revenue_type"])
        eligible = bool(item.get("evaluation_eligible", item.get("evaluation_status") == "eligible"))
        previous = eligibility.setdefault(key, {"eligible": False, "forecast_only": False, "split": item.get("split")})
        previous["eligible"] |= eligible
        previous["forecast_only"] |= bool(item.get("has_forecast_only_task", not eligible))
    choices = {}
    for item in available_records:
        channel = item.get("revenue_type")
        if channel not in {"sales", "lease"}:
            continue
        run_id, key = str(item["forecast_run_id"]), str(item["target_game_key"])
        status = eligibility.get((run_id, key, channel))
        if view == "test" and not (status and status["eligible"] and status["split"] == "test"):
            continue
        if view == "forecast_only" and status and not status["forecast_only"]:
            continue
        identity = (key, channel)
        choice = choices.setdefault(identity, {"target_game_key": key, "target_game": str(item["target_game"]),
                                               "revenue_type": channel, "run_ids": set()})
        choice["run_ids"].add(run_id)
    result = []
    for identity, choice in choices.items():
        label = f"{choice['target_game']} | {'Sale' if choice['revenue_type'] == 'sales' else 'Rent'}"
        result.append((label, identity, {**choice, "run_ids": sorted(choice["run_ids"])}))
    return sorted(result, key=lambda item: (item[0].casefold(), item[1]))


def demand_curve_buttons():
    import ipywidgets as widgets
    controls = {key: widgets.RadioButtons(options=[("On", True), ("Off", False)],
                                           value=(key == "baseline"), description=label,
                                           style={"description_width": "initial"},
                                           layout=widgets.Layout(width="165px"))
                for key, (_, label) in DEMAND_CURVES.items()}
    return controls, widgets.Box(list(controls.values()), layout=widgets.Layout(display="flex", flex_flow="row wrap"))


def launch_forecast_review(spark, forecast_tables, evaluation_tables, metrics, history_rows=30):
    """Display experiment → game/channel → run selectors; never writes/recomputes.

    Required forecast_tables: runs,demand,performance,neighbors. Optional saved
    evaluation_tables: runs,cohorts,eligibility,metrics. Actuals always come from
    the selected run's latest completed evaluation cohort, never 06 placeholders.
    """
    import json
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    from pyspark.sql import functions as F

    if not spark.catalog.tableExists(forecast_tables["runs"]):
        print("No saved forecast status table. Run notebook 06 first.")
        return None
    runs_df = spark.table(forecast_tables["runs"])
    status_column = "processing_status" if "processing_status" in runs_df.columns else "status"
    columns = [name for name in ["forecast_run_id", "experiment_id", "experiment_name", "data_revision",
                                "started_at_utc", "completed_at_utc", "tracking_status", "config_json"] if name in runs_df.columns]
    runs = [row.asDict() for row in runs_df.where(F.col(status_column) == "complete").select(*columns).collect()]
    if not runs:
        print("No completed forecasts. Run notebook 06 first.")
        return None
    run_meta = {row["forecast_run_id"]: row for row in runs}
    experiments = {}
    for row in runs:
        settings = json.loads(row.get("config_json") or "{}")
        name = row.get("experiment_name") or row["experiment_id"]
        spec = settings.get("named_experiment")
        if spec:
            weights = ", ".join(f"{w:g}" for w in spec["rank_weights"])
            row["weight_label"] = f"Rank weights [{weights}]"
            row["review_experiment"] = "named:" + spec["name"]
            experiments[row["review_experiment"]] = spec["name"]
        else:
            multiplier = settings.get("top_neighbor_multiplier", 1.0)
            row["weight_label"] = "Equal weights" if multiplier == 1.0 else f"Highest neighbor ×{multiplier:g}"
            row["review_experiment"] = "legacy:" + row["experiment_id"]
            base_name = name.rsplit(" | ", 1)[0] if multiplier != 1.0 else name
            experiments[row["review_experiment"]] = f"{base_name} | {row['experiment_id'][-8:]}"
    experiment_options = sorted([(name, key) for key, name in experiments.items()], key=lambda x: x[0].casefold())
    view = widgets.Dropdown(options=[("Evaluated test games", "test"),
                                    ("Forecast only / unevaluated", "forecast_only"), ("All saved games", "all")],
                            value="test", description="View:", layout=widgets.Layout(width="550px"))
    experiment = widgets.Dropdown(options=experiment_options, description="Experiment:", layout=widgets.Layout(width="750px"))
    game = widgets.Dropdown(options=[], description="Game/channel:", layout=widgets.Layout(width="750px"))
    run = widgets.Dropdown(options=[], description="Forecast run:", layout=widgets.Layout(width="750px"))
    output, notice = widgets.Output(), widgets.Output()
    curve_switches, curve_box = demand_curve_buttons()
    demand_output = widgets.Output()
    state = {"busy": False, "choices": {}, "evaluations": {}, "available": [], "eligibility": []}

    def exists(label):
        return bool(evaluation_tables.get(label)) and spark.catalog.tableExists(evaluation_tables[label])

    def read_evaluation_index(run_ids):
        latest = {}
        if not exists("runs"):
            return latest, []
        frame = spark.table(evaluation_tables["runs"])
        status = "processing_status" if "processing_status" in frame.columns else "status"
        columns = [name for name in ["forecast_run_id", "evaluation_id", "cohort_id", "completed_at_utc", "updated_at_utc"] if name in frame.columns]
        if "metadata_json" in frame.columns:
            columns.append(F.get_json_object("metadata_json", "$.diagnostics.training_bias").alias("training_bias"))
        records = [row.asDict() for row in frame.where((F.col(status) == "complete") & F.col("forecast_run_id").isin(run_ids)).select(*columns).collect()]
        records.sort(key=lambda row: (row.get("completed_at_utc") or row.get("updated_at_utc") or "", row["evaluation_id"]), reverse=True)
        for item in records:
            latest.setdefault(item["forecast_run_id"], item)
        if not latest or not exists("eligibility"):
            return latest, []
        cohort_ids = sorted({item["cohort_id"] for item in latest.values()})
        eligibility = spark.table(evaluation_tables["eligibility"]).where(F.col("cohort_id").isin(cohort_ids))
        eligible = (F.coalesce(F.col("evaluation_eligible").cast("boolean"), F.col("evaluation_status") == "eligible") if "evaluation_eligible" in eligibility.columns
                    else F.col("evaluation_status") == "eligible")
        grouped = eligibility.groupBy("cohort_id", "target_game_key", "revenue_type").agg(
            F.max(eligible.cast("int")).alias("_any_eligible"), F.min(eligible.cast("int")).alias("_all_eligible"),
            F.first("split", ignorenulls=True).alias("split"))
        by_cohort = {}
        for row in grouped.collect():
            item = row.asDict()
            by_cohort.setdefault(item["cohort_id"], []).append({**item, "evaluation_eligible": item["_any_eligible"] == 1,
                                                              "has_forecast_only_task": item["_all_eligible"] != 1})
        return latest, [{**item, "forecast_run_id": run_id} for run_id, evaluation in latest.items()
                        for item in by_cohort.get(evaluation["cohort_id"], [])]

    def select_frame(table, run_id, key, channel):
        if not spark.catalog.tableExists(table):
            return None
        frame = spark.table(table).where((F.col("forecast_run_id") == run_id) & (F.col("target_game_key") == key))
        if "revenue_type" in frame.columns:
            frame = frame.where(F.col("revenue_type") == channel)
        elif "own_status" in frame.columns:
            frame = frame.where(F.col("own_status") == ("owned" if channel == "sales" else "leased"))
        return frame

    def actual_points(evaluation, key, channel):
        if not evaluation or not exists("cohorts"):
            return pd.DataFrame()
        frame = spark.table(evaluation_tables["cohorts"]).where(
            (F.col("cohort_id") == evaluation["cohort_id"]) & (F.col("target_game_key") == key)
            & (F.col("revenue_type") == channel))
        return frame.toPandas()

    def numeric(frame, name):
        return pd.to_numeric(frame[name], errors="coerce").to_numpy(dtype=float) if name in frame else np.full(len(frame), np.nan)

    def panel(axis, forecasts, points, value, lower, upper, title, ylabel, curves=None):
        axis.set_title(title, loc="left")
        axis.set_xlabel("Forecast month")
        axis.set_ylabel(ylabel)
        axis.grid(alpha=0.2)
        if not forecasts.empty and not points.empty and "forecast_month" in points:
            forecasts = forecasts.copy()
            references = points.drop_duplicates("month_index").set_index("month_index")
            if "forecast_month" in forecasts:
                dates = pd.to_datetime(forecasts["forecast_month"], errors="coerce")
            elif "forecast_yearmonth" in forecasts:
                dates = pd.to_datetime(forecasts["forecast_yearmonth"].astype("string") + "01", format="%Y%m%d", errors="coerce")
            else:
                dates = pd.Series(pd.NaT, index=forecasts.index)
            has_label = forecasts["month_index"].isin(references.index)
            reference_dates = pd.to_datetime(forecasts["month_index"].map(references["forecast_month"]), errors="coerce")
            mismatch = has_label & (dates.isna() | reference_dates.isna() | dates.ne(reference_dates))
            if "prediction_date" in references and "prediction_date" in forecasts:
                origins = pd.to_datetime(forecasts["prediction_date"], errors="coerce")
                expected = pd.to_datetime(forecasts["month_index"].map(references["prediction_date"]), errors="coerce")
                mismatch |= has_label & (origins.isna() | expected.isna() | origins.ne(expected))
            for name in [value, lower, upper, *(DEMAND_CURVES[key][0] for key in (curves or []))]:
                if name in forecasts:
                    forecasts.loc[mismatch, name] = np.nan
            if mismatch.any():
                axis.text(0.02, 0.69, f"{int(mismatch.sum())} calendar/origin mismatch(es) excluded", transform=axis.transAxes, fontsize=9)
        if not forecasts.empty:
            x, y = numeric(forecasts, "month_index"), numeric(forecasts, value)
            show_baseline = curves is None or "baseline" in curves
            if show_baseline and np.isfinite(y).any():
                axis.plot(x, y, marker="o", color="#176B87", label="Neighbor mean" if curves is not None else "Saved forecast")
            colors = {"baseline": "#176B87", "linear": "#bb5422", "poly2": "#7952a2", "log": "#227148"}
            for key in (curves or []):
                if key == "baseline":
                    continue
                column, label = DEMAND_CURVES[key]
                fit = numeric(forecasts, column)
                if np.isfinite(fit).any():
                    base = key.removesuffix("_train_bias")
                    axis.plot(x, fit, color=colors[base], label=label, linewidth=2,
                              linestyle="--" if base != key else "-")
            if show_baseline and lower in forecasts and upper in forecasts:
                axis.fill_between(x, numeric(forecasts, lower), numeric(forecasts, upper), color="#176B87", alpha=0.15, label="Neighbor variation (±2 SD)")
            if curves is None and not np.isfinite(y).any():
                axis.text(0.02, 0.92, "No usable saved forecast", transform=axis.transAxes, va="top")
        if not points.empty:
            points = points.sort_values("month_index")
            axis.plot(numeric(points, "month_index"), numeric(points, "actual"), marker="D", linestyle="--", color="#222222", label="Frozen observed actual")
        else:
            axis.text(0.02, 0.80, "No eligible actuals in this saved cohort", transform=axis.transAxes, va="top", fontsize=9)
        handles, _ = axis.get_legend_handles_labels()
        if handles:
            axis.legend(fontsize=8)

    def redraw_demand(change=None):
        selected = state.get("demand_panel")
        if selected is None:
            return
        forecasts, actuals, label = selected
        visible = [key for key, button in curve_switches.items() if button.value and button.layout.display != "none"]
        with demand_output:
            clear_output(wait=True)
            figure, axis = plt.subplots(figsize=(12, 3.8))
            panel(axis, forecasts, actuals, "forecast_value", "low_band_2std", "high_band_2std", label, label, visible)
            figure.tight_layout()
            display(figure)
            plt.close(figure)
            unavailable = [name for key in visible for col, name in [DEMAND_CURVES[key]]
                           if not np.isfinite(numeric(forecasts, col)).any()]
            if unavailable:
                print("No saved values for this game: " + ", ".join(unavailable))

    def render(change=None):
        if state["busy"]:
            return
        state["demand_panel"] = None
        with output:
            clear_output(wait=True)
            if game.value is None or run.value is None:
                print("No choices in this view. Run 07 for evaluated test games, or select forecast-only/all saved games.")
                return
            try:
                context = state["choices"][game.value]
                key, channel, run_id = context["target_game_key"], context["revenue_type"], run.value
                evaluation = state["evaluations"].get(run_id)
                demand_frame = select_frame(forecast_tables["demand"], run_id, key, channel)
                perf_frame = select_frame(forecast_tables["performance"], run_id, key, channel)
                demand = demand_frame.orderBy("month_index").toPandas() if demand_frame is not None else pd.DataFrame()
                performance = perf_frame.orderBy("month_index").toPandas() if perf_frame is not None else pd.DataFrame()
                points = actual_points(evaluation, key, channel)
                print(f"{context['target_game']} | {'Sale' if channel == 'sales' else 'Rent'} | {run_id}")
                print(f"Forecast tracking: {run_meta[run_id].get('tracking_status', 'unknown')}")
                print(f"Evaluation: {evaluation['evaluation_id']} | Cohort: {evaluation['cohort_id']}" if evaluation else "No completed evaluation for this forecast; actuals are not inferred from 06 placeholders.")
                if evaluation and exists("eligibility"):
                    labels = spark.table(evaluation_tables["eligibility"]).where((F.col("cohort_id") == evaluation["cohort_id"])
                        & (F.col("target_game_key") == key) & (F.col("revenue_type") == channel))
                    columns = [name for name in ["forecast_type", "metric_name", "split", "evaluation_status", "eligibility_reason", "observed_months", "actual_bracket"] if name in labels.columns]
                    display(labels.select(*columns).toPandas())
                commercial = points.loc[points["forecast_type"].eq("demand")] if not points.empty and "forecast_type" in points else pd.DataFrame()
                label = str(demand["curve_label"].iloc[0]) if not demand.empty and "curve_label" in demand else ("Cumulative sales" if channel == "sales" else "Monthly rental records")
                calibration = json.loads((evaluation or {}).get("training_bias") or "{}")
                for curve, (column, _) in BASE_DEMAND_CURVES.items():
                    bias_curve_key = curve + "_train_bias"
                    available = column in demand and curve in calibration.get("offsets", {})
                    curve_switches[bias_curve_key].layout.display = "flex" if available else "none"
                    if available:
                        demand[DEMAND_CURVES[bias_curve_key][0]] = apply_training_bias(demand, calibration, curve, demand[column])
                state["demand_panel"] = (demand, commercial, label)
                display(curve_box, demand_output)
                redraw_demand()
                if metrics:
                    figure, axes = plt.subplots(len(metrics), 1, figsize=(12, 3.2 * len(metrics)), squeeze=False)
                    for metric, axis in zip(metrics, axes[:, 0]):
                        actual = points.loc[points["forecast_type"].eq("performance") & points["metric_name"].eq(metric)] if not points.empty and "forecast_type" in points else pd.DataFrame()
                        panel(axis, performance, actual, metric + "_mean", metric + "_lower_2std", metric + "_higher_2std", metric.replace("_", " "), "Index")
                    figure.tight_layout(); display(figure); plt.close(figure)
                if evaluation and exists("metrics"):
                    frame = with_curve_name(spark.table(evaluation_tables["metrics"]))
                    if {"evaluation_id", "target_game_key", "revenue_type"}.issubset(frame.columns):
                        frame = frame.where((F.col("evaluation_id") == evaluation["evaluation_id"]) & (F.col("target_game_key") == key) & (F.col("revenue_type") == channel))
                        names = [name for name in ["forecast_type", "metric_name", "curve_name", "evaluation_split", "valid_months", "expected_months", "actual_bracket", "bracket", "mae", "rmse", "bias", "directional_accuracy", "forecast_coverage", "prediction_coverage", "coverage"] if name in frame.columns]
                        display(frame.select(*names).toPandas())

                # The neighbor panel also works for runs without an evaluation.
                neighbors = select_frame(forecast_tables["neighbors"], run_id, key, channel)
                prediction_date = demand.iloc[0]["prediction_date"] if not demand.empty else None
                show_neighbor_review(spark, neighbors, forecast_tables["demand"], run_meta[run_id],
                                     channel, prediction_date, label)
            except Exception as error:
                print(f"Review could not load this selection: {type(error).__name__}: {error}")

    def rebuild_runs(change=None):
        if state["busy"]:
            return
        state["busy"] = True
        try:
            choice = state["choices"].get(game.value)
            options = [] if not choice else [(f"{run_meta[key]['weight_label']} | {run_meta[key].get('completed_at_utc') or run_meta[key].get('started_at_utc') or ''} | {key}", key)
                                            for key in sorted(choice["run_ids"], key=lambda item: (run_meta[item].get("completed_at_utc") or run_meta[item].get("started_at_utc") or "", item), reverse=True)]
            run.options = options
        finally:
            state["busy"] = False
        render()

    def rebuild_games(change=None):
        if state["busy"]:
            return
        state["busy"] = True
        try:
            run_ids = [row["forecast_run_id"] for row in runs if row["review_experiment"] == experiment.value]
            available = spark.table(forecast_tables["demand"]).where(F.col("forecast_run_id").isin(run_ids)).select(
                "forecast_run_id", "target_game_key", "target_game", "revenue_type").distinct()
            state["available"] = [row.asDict() for row in available.collect()]
            state["evaluations"], state["eligibility"] = read_evaluation_index(run_ids)
            choices = build_saved_review_options(state["available"], state["eligibility"], view.value)
            state["choices"] = {identity: context for _, identity, context in choices}
            game.options = [(label, identity) for label, identity, _ in choices]
            with notice:
                clear_output(wait=True)
                print(f"{len(choices)} game/channel choices. Actuals use each forecast's latest completed evaluation cohort.")
        except Exception as error:
            state["choices"], game.options, run.options = {}, [], []
            with notice:
                clear_output(wait=True)
                print(f"Review choices unavailable: {type(error).__name__}: {error}")
        finally:
            state["busy"] = False
        rebuild_runs()

    experiment.observe(rebuild_games, names="value")
    view.observe(rebuild_games, names="value")
    game.observe(rebuild_runs, names="value")
    run.observe(render, names="value")
    for button in curve_switches.values():
        button.observe(redraw_demand, names="value")
    display(widgets.VBox([view, experiment, game, run, notice, output]))
    rebuild_games()
    return {"view": view, "experiment": experiment, "game": game, "run": run, "curves": curve_switches, "output": output}


In [0]:
review_config = load_evaluation_config(CONFIG_PATH)
review_controls = launch_forecast_review(
    spark, review_config["forecast"]["outputs"], review_config["outputs"],
    review_config["settings"]["performance_metrics"], review_config["monitoring"]["history_rows"],
)


## Compare approaches and inspect failures

Choose a measure and population, such as cumulative sales or leased-game coin-in index. Each comparison uses one frozen cohort and evaluation policy.

- **Approach comparison** shows notebook 07's saved scores, coverage, and validation rank. Shared MAE compares the selected approaches on identical observed game-months. Change the approaches included to compare a pair directly.
- **Failure analysis** shows the selected run's errors by actual bracket and forecast month, the worst games and individual misses, and missing predictions with reasons.
- **Game curves** overlays each selected approach on the same game's observed values. The table below the chart gives the exact values for the eligible evaluation months.

Use validation for choosing an approach. Held-out test shows how that choice performed on later games. Sales, rentals and Eilers indices have different units, so each has its own comparison.

For demand, switch each curve on or off independently. The comparison and failure panels use the selected run-and-curve series. Validation ranks include all scored candidates for the task. Shared MAE uses the game-months all visible series have in common.


In [0]:
import json
import numpy as np
import pandas as pd

REVIEW_TASK = ["forecast_type", "metric_name", "revenue_type", "own_status"]
REVIEW_POINT = ["target_game_key", "month_index"]


def align_review_predictions(labels, predictions, run_ids):
    """Keep every eligible observed game-month, including absent or misaligned forecasts."""
    frames = []
    for run_id in run_ids:
        forecast = predictions.loc[predictions["forecast_run_id"].eq(run_id)].drop(columns="forecast_run_id")
        points = labels.merge(forecast, on=REVIEW_POINT, how="left", validate="one_to_one", indicator=True)
        expected_month = pd.to_datetime(points["forecast_month"]).dt.to_period("M")
        saved_month = pd.to_datetime(points["saved_month"], errors="coerce").dt.to_period("M")
        same_origin = pd.to_datetime(points["prediction_date"]).eq(pd.to_datetime(points["saved_origin"], errors="coerce"))
        value = pd.to_numeric(points["prediction"], errors="coerce")
        status = points["forecast_status"].fillna("no_value_for_metric").replace("ready", "no_value_for_metric")
        points["reason"] = np.select(
            [points["_merge"].eq("left_only"), ~(expected_month.eq(saved_month) & same_origin), np.isfinite(value)],
            ["missing_forecast_row", "calendar_or_origin_mismatch", "scored"], default=status.to_numpy(dtype=str))
        points["prediction"] = value.where(points["reason"].eq("scored"))
        points["error"] = points["prediction"] - points["actual"]
        points["abs_error"] = points["error"].abs()
        points["forecast_run_id"] = run_id
        frames.append(points.drop(columns=["_merge", "saved_month", "saved_origin", "forecast_status"]))
    return pd.concat(frames, ignore_index=True)


def review_task_rows(frame, task):
    from pyspark.sql import functions as F

    for column, value in zip(REVIEW_TASK, task):
        frame = frame.where(F.col(column) == value)
    return frame


def read_review_scores(spark, tables, units, cohort_id, policy, task, split):
    from pyspark.sql import functions as F

    return review_task_rows(with_curve_name(spark.table(tables["metrics"])), task).where(
        (F.col("cohort_id") == cohort_id) & (F.col("policy_hash") == policy)
        & F.col("evaluation_id").isin(units["evaluation_id"].tolist())
        & F.col("result_level").isin("overall", "bracket") & F.col("evaluation_split").isin("validation", split)
    ).select("forecast_run_id", "curve_name", "result_level", "evaluation_split", "bracket", "scored_games", "expected_games",
             "valid_months", "expected_months", "forecast_coverage", "mae", "rmse", "bias", "directional_accuracy",
             "eligible_for_selection").toPandas()


def read_review_points(spark, tables, forecast_tables, series, cohort_id, task, split):
    from pyspark.sql import functions as F

    run_ids = series["forecast_run_id"].unique().tolist()
    labels = review_task_rows(spark.table(tables["cohorts"]), task).where(
        (F.col("cohort_id") == cohort_id) & (F.col("evaluation_split") == split)
    ).select(*REVIEW_POINT, "target_game", "actual", "actual_bracket", "forecast_month", "prediction_date")
    observed = labels.toPandas()
    if observed.empty:
        return pd.DataFrame()

    kind, metric, channel, ownership = task
    source = spark.table(forecast_tables["demand" if kind == "demand" else "performance"])
    source = source.where(F.col("forecast_run_id").isin(run_ids) & (F.col("revenue_type") == channel)
                          & (F.col("own_status") == ownership))
    if kind == "demand":
        source = source.where(F.col("metric_name") == metric)
        neighbor_count = "neighbor_count"
        base_curves = {key.removesuffix("_train_bias") for key in series["curve_name"]}
        columns = {key: BASE_DEMAND_CURVES[key][0] for key in sorted(base_curves)}
    else:
        neighbor_count = metric + "_neighbor_count"
        columns = {"baseline": metric + "_mean"}
    if "forecast_month" in source.columns:
        month = F.col("forecast_month")
    else:
        month = F.expr("try_cast(concat(substr(cast(forecast_yearmonth as string), 1, 4), '-', "
                       "substr(cast(forecast_yearmonth as string), 5, 2), '-01') as date)")
    # Expected labels stay on the left during alignment. Missing 06 rows count against coverage.
    projected = source.join(labels.select(*REVIEW_POINT), REVIEW_POINT, "left_semi").select(
        "forecast_run_id", *REVIEW_POINT, month.alias("saved_month"), F.col("prediction_date").alias("saved_origin"),
        *[(F.col(column) if column in source.columns else F.lit(None).cast("double")).alias(key)
          for key, column in columns.items()], F.col(neighbor_count).alias("neighbors"),
        F.col("forecast_eligibility_status").alias("forecast_status")).toPandas()
    aligned = []
    for curve, units in series.groupby("curve_name", sort=False):
        base = curve.removesuffix("_train_bias")
        values = projected.drop(columns=list(columns)).assign(prediction=projected[base])
        if base != curve:
            values["revenue_type"] = channel
            for unit in units.itertuples(index=False):
                mask = values["forecast_run_id"].eq(unit.forecast_run_id)
                batch = values.loc[mask]
                calibration = json.loads(unit.training_bias or "{}")
                values.loc[mask, "prediction"] = apply_training_bias(
                    batch, calibration, base, batch["prediction"], origin_column="saved_origin")
            values = values.drop(columns="revenue_type")
        if curve != "baseline":
            values["forecast_status"] = "missing_curve_value"
        points = align_review_predictions(observed, values, units["forecast_run_id"].tolist())
        points["series_id"] = points["forecast_run_id"].map(units.set_index("forecast_run_id")["series_id"])
        aligned.append(points)
    return pd.concat(aligned, ignore_index=True) if aligned else pd.DataFrame()


def review_scoreboard(units, saved, points, included, split):
    board = units.loc[units["series_id"].isin(included), ["series_id", "approach"]].copy()
    totals = saved.loc[saved["result_level"].eq("overall")]
    validation = totals.loc[totals["evaluation_split"].eq("validation") & totals["eligible_for_selection"].eq(True)]
    rank = validation.sort_values(["mae", "rmse", "forecast_run_id", "curve_name"]).reset_index(drop=True)
    rank = rank.assign(validation_rank=np.arange(1, len(rank) + 1))
    board = board.merge(rank[["series_id", "validation_rank"]], on="series_id", how="left", validate="one_to_one")
    board = board.merge(totals.loc[totals["evaluation_split"].eq(split)], on="series_id", how="left", validate="one_to_one")
    count, games, expected = 0, 0, 0
    board["shared_mae"] = np.nan
    if points is not None and not points.empty:
        matrix = points.loc[points["series_id"].isin(included)].pivot(
            index=REVIEW_POINT, columns="series_id", values="abs_error").reindex(columns=included)
        shared = matrix.dropna(how="any")
        board["shared_mae"] = board["series_id"].map(shared.mean())
        count, games, expected = len(shared), shared.index.get_level_values("target_game_key").nunique(), len(matrix)
    board["validation_rank"] = board["validation_rank"].astype("Int64")
    order = ["validation_rank", "shared_mae", "approach"] if split == "validation" else ["validation_rank", "approach"]
    board = board.sort_values(order, na_position="last")
    return board, count, games, expected


def launch_accuracy_review(spark, tables, forecast_tables):
    import matplotlib.pyplot as plt
    import ipywidgets as widgets
    from IPython.display import HTML, clear_output, display
    from pyspark.sql import functions as F

    required = ["runs", "eligibility", "cohorts", "metrics"]
    if any(not spark.catalog.tableExists(tables[k]) for k in required):
        print("Run notebook 07 to populate the evaluation tables for this comparison.")
        return
    index = (spark.table(tables["runs"]).where(F.col("processing_status") == "complete")
             .select("cohort_id", "policy_hash", "data_revision", "completed_at_utc", "evaluation_id",
                     "forecast_run_id", "experiment_name",
                     F.get_json_object("metadata_json", "$.diagnostics.training_bias").alias("training_bias")).toPandas()
             .sort_values(["completed_at_utc", "evaluation_id"], ascending=False)
             .drop_duplicates(["cohort_id", "policy_hash", "forecast_run_id"]))
    if index.empty:
        print("No completed evaluations are available for comparison.")
        return
    groups = index.drop_duplicates(["cohort_id", "policy_hash"])
    cohort = widgets.Dropdown(description="Cohort", options=[
        (f"{r.data_revision} | {r.cohort_id} | policy {r.policy_hash[-8:]}", (r.cohort_id, r.policy_hash))
        for r in groups.itertuples()])
    task = widgets.Dropdown(description="Measure")
    split = widgets.ToggleButtons(description="Split", options=[("Validation", "validation"), ("Held-out test", "test")])
    view = widgets.ToggleButtons(options=["Approach comparison", "Failure analysis", "Game curves"])
    approaches = widgets.SelectMultiple(description="Approaches", rows=6)
    curve_switches, curve_box = demand_curve_buttons()
    subset = widgets.Accordion(children=[approaches], selected_index=None)
    subset.set_title(0, "Choose approaches")
    focus = widgets.Dropdown(description="Inspect series")
    game = widgets.Dropdown(description="Game")
    for widget in (task, cohort, approaches, focus, game):
        widget.layout.width = "min(100%, 850px)"
    output = widgets.Output()
    state = {"busy": False, "units": None, "data": None}

    def show_table(frame):
        display(HTML('<div style="max-width:100%;overflow:auto;max-height:450px">'
                     + frame.to_html(index=False, border=0, escape=True, na_rep="—",
                                     float_format=lambda x: f"{x:,.4g}").replace("&lt;NA&gt;", "—") + "</div>"))

    fields = {"scored_games": "Games scored", "expected_games": "Games expected", "valid_months": "Months scored",
              "expected_months": "Months expected", "mae": "MAE", "rmse": "RMSE", "bias": "Bias",
              "directional_accuracy": "Direction %", "forecast_coverage": "Coverage %"}

    def score_table(frame, leading):
        names = {**leading, **fields}
        frame = frame[list(names)].copy()
        for col in ("directional_accuracy", "forecast_coverage"):
            frame[col] = 100 * frame[col]
        show_table(frame.rename(columns=names))

    def visible_series():
        series = state.get("series")
        if series is None:
            return []
        demand = task.value and task.value[0] == "demand"
        curves = [key for key, button in curve_switches.items() if button.value] if demand else ["baseline"]
        return series.loc[series["forecast_run_id"].isin(approaches.value)
                          & series["curve_name"].isin(curves), "series_id"].tolist()

    def render(_=None):
        if state["busy"]:
            return
        focus.layout.display = "" if view.value == "Failure analysis" else "none"
        game.layout.display = "" if view.value == "Game curves" else "none"
        with output:
            clear_output(wait=True)
            if state["data"] is None:
                print(state.get("notice", "No comparable results for this cohort."))
                return
            saved, points = state["data"]
            if points is None:
                print(state["notice"])
            elif points.empty:
                print("No eligible observed game-months for this measure and split in the selected cohort.")
                return
            included = visible_series()
            if not approaches.value:
                print("Select at least one approach.")
                return
            if not included and view.value != "Game curves":
                print("Enable a curve with saved scores for this selection.")
                return
            units = state["series"]
            labels = units.set_index("series_id")["approach"]
            if view.value == "Approach comparison":
                board, months, games, expected = review_scoreboard(units, saved, points, included, split.value)
                if points is not None:
                    print(f"Shared MAE uses the same {months} of {expected} observed game-months ({games} games) "
                          f"across the {len(included)} selected forecast series.")
                if points is not None and not months:
                    print("The selected approaches have no scored months in common. Choose a subset to compare overlap.")
                if board["validation_rank"].isna().all():
                    print("None of these approaches meets notebook 07's validation selection rule.")
                print("Validation rank uses all completed approaches in this cohort, ordered by MAE, then RMSE, run ID, and curve name. "
                      "Positive bias means overprediction.")
                score_table(board, {"approach": "Approach", "validation_rank": "Validation rank", "shared_mae": "Shared MAE"})
            elif view.value == "Failure analysis":
                run_id = focus.value
                bracket = saved.loc[saved["series_id"].eq(run_id) & saved["result_level"].eq("bracket")
                                    & saved["evaluation_split"].eq(split.value)]
                display(HTML("<h4>Error by actual bracket</h4>"))
                if not bracket.empty:
                    score_table(bracket.sort_values("bracket"), {"bracket": "Actual bracket"})
                else:
                    print("No saved bracket scores for this run and split.")
                if points is None:
                    return
                data = points.loc[points["series_id"].eq(run_id)]
                horizon = data.groupby("month_index", sort=True).agg(
                    expected=("actual", "size"), scored=("prediction", "count"), mae=("abs_error", "mean"), bias=("error", "mean"))
                horizon["coverage"] = 100 * horizon.scored / horizon.expected
                print("Horizon counts include game-months with eligible actuals. Positive bias means overprediction.")
                if not horizon.empty:
                    horizon = horizon.reindex(range(1, int(horizon.index.max()) + 1))
                    fig, axes = plt.subplots(2, 1, figsize=(9, 4.8), sharex=True, constrained_layout=True)
                    axes[0].plot(horizon.index, horizon.mae, "o-", label="MAE")
                    axes[0].plot(horizon.index, horizon.bias, "s--", label="Bias")
                    axes[0].axhline(0, linewidth=0.7, color="gray")
                    axes[0].set_ylabel("Forecast error")
                    axes[0].legend()
                    axes[1].bar(horizon.index, horizon.coverage, color="#277688")
                    axes[1].set(ylabel="Coverage (%)", xlabel="Forecast month", ylim=(0, 105))
                    axes[1].set_xticks(horizon.index)
                    display(fig)
                    plt.close(fig)
                display(HTML("<h4>Games with the largest average error</h4>"))
                by_game = data.groupby(["target_game_key", "target_game"]).agg(
                    mae=("abs_error", "mean"), bias=("error", "mean"), scored=("prediction", "count"), expected=("actual", "size"))
                ranked = by_game.dropna(subset=["mae"]).nlargest(10, "mae").reset_index()
                if not ranked.empty:
                    show_table(ranked[["target_game", "mae", "bias", "scored", "expected"]].set_axis(
                        ["Game", "MAE", "Bias", "Months scored", "Months expected"], axis=1))
                else:
                    print("No games with scored predictions.")
                display(HTML("<h4>Largest individual misses</h4>"))
                worst = data.dropna(subset=["abs_error"]).nlargest(15, "abs_error")
                if worst.empty:
                    print("No scored forecast months for this approach.")
                else:
                    show_table(worst[["target_game", "actual_bracket", "month_index", "forecast_month", "actual", "prediction",
                                      "error", "neighbors"]].set_axis(
                        ["Game", "Bracket", "Month", "Calendar month", "Actual", "Forecast", "Forecast − actual", "Contributing neighbors"], axis=1))
                missing = data.loc[data["reason"].ne("scored")]
                display(HTML("<h4>Missing predictions</h4>"))
                if missing.empty:
                    print("Every eligible game-month has a scored prediction.")
                else:
                    report = missing.groupby("reason").agg(games=("target_game_key", "nunique"), months=("actual", "size"))
                    show_table(report.sort_values("months", ascending=False).reset_index().rename(
                        columns={"reason": "Reason", "games": "Affected games", "months": "Missing months"}))
                    by_game = missing.groupby(["target_game_key", "target_game"]).agg(
                        months=("actual", "size"), reasons=("reason", lambda x: ", ".join(sorted(set(x)))))
                    show_table(by_game.nlargest(10, "months").reset_index()[["target_game", "months", "reasons"]].set_axis(
                        ["Game", "Missing months", "Reasons"], axis=1))
            else:
                if points is None:
                    return
                data = points.loc[points["target_game_key"].eq(game.value)]
                if data.empty:
                    return
                truth = data.drop_duplicates("month_index").set_index("month_index")["actual"]
                months = range(1, int(truth.index.max()) + 1)
                curves = data.loc[data["series_id"].isin(included)].pivot(index="month_index", columns="series_id", values="prediction").reindex(
                    index=months, columns=included)
                fig, axis = plt.subplots(figsize=(12, 4.3), constrained_layout=True)
                for run_id in included:
                    if curves[run_id].notna().any():
                        axis.plot(curves.index, curves[run_id], marker=".", alpha=.8, label=labels[run_id])
                axis.plot(months, truth.reindex(months), "kD--", linewidth=2, label="Observed actual")
                axis.set(title=str(data["target_game"].iloc[0]), xlabel="Forecast month", ylabel=task.value[1].replace("_", " "))
                axis.legend(bbox_to_anchor=(1, 1), loc="upper left", fontsize=8)
                display(fig)
                plt.close(fig)
                curves = curves.rename(columns=labels.to_dict())
                curves.insert(0, "Actual", truth)
                show_table(curves.rename_axis("Month").reset_index())

    def update_approaches(_=None):
        if state["busy"]:
            return
        state["busy"] = True
        try:
            chosen = set(visible_series())
            previous = focus.value
            series = state.get("series")
            focus.options = [] if series is None else list(series.loc[series["series_id"].isin(chosen),
                ["approach", "series_id"]].itertuples(index=False, name=None))
            if previous in chosen:
                focus.value = previous
        finally:
            state["busy"] = False
        render()

    def load_slice(_=None):
        if state["busy"]:
            return
        state["busy"] = True
        state["data"] = None
        state["series"] = None
        curve_box.layout.display = "flex" if task.value and task.value[0] == "demand" else "none"
        try:
            if task.value is None:
                state["notice"] = "No evaluation tasks recorded for this cohort."
                game.options = []
            else:
                cohort_id, policy = cohort.value
                saved = read_review_scores(spark, tables, state["units"], cohort_id, policy, task.value, split.value)
                series = saved[["forecast_run_id", "curve_name"]].drop_duplicates().merge(
                    state["units"][["forecast_run_id", "approach", "training_bias"]], on="forecast_run_id", validate="many_to_one")
                if series.empty:
                    state["notice"] = "No saved validation or selected-split scores for this measure."
                    game.options = []
                    with output:
                        clear_output(wait=True)
                        print(state["notice"])
                    return
                series["series_id"] = series["forecast_run_id"] + "|" + series["curve_name"]
                if task.value[0] == "demand":
                    series["approach"] = series["approach"] + " | " + series["curve_name"].map(
                        {key: label for key, (_, label) in DEMAND_CURVES.items()})
                for key, button in curve_switches.items():
                    button.layout.display = "flex" if key in set(series["curve_name"]) else "none"
                state["series"] = series
                saved = saved.merge(series[["forecast_run_id", "curve_name", "series_id"]],
                                    on=["forecast_run_id", "curve_name"], validate="many_to_one")
                try:
                    points = read_review_points(spark, tables, forecast_tables, series, cohort_id, task.value, split.value)
                    totals = saved.loc[saved["result_level"].eq("overall") & saved["evaluation_split"].eq(split.value)]
                    if not totals.empty:
                        coverage = (points.groupby("series_id")["prediction"].agg(["size", "count"])
                                    if not points.empty else pd.DataFrame(columns=["size", "count"]))
                        for row in totals.itertuples():
                            if (row.series_id not in coverage.index or
                                tuple(coverage.loc[row.series_id]) != (row.expected_months, row.valid_months)):
                                raise ValueError(f"Point counts differ from saved 07 scores for {row.forecast_run_id}.")
                except Exception as exc:
                    points = None
                    state["notice"] = f"Saved metrics are available. Monthly diagnostics unavailable: {type(exc).__name__}: {exc}"
                state["data"] = saved, points
                previous = game.value
                game.options = [] if points is None or points.empty else list(points[["target_game", "target_game_key"]].drop_duplicates()
                                                           .sort_values("target_game").itertuples(index=False, name=None))
                if previous is not None and previous in {key for _, key in game.options}:
                    game.value = previous
        except Exception as exc:
            state["notice"] = f"Could not load the selected evaluation: {type(exc).__name__}: {exc}"
            game.options = []
        finally:
            state["busy"] = False
        update_approaches()

    def change_cohort(_=None):
        cohort_id, policy = cohort.value
        state["busy"] = True
        try:
            units = index.loc[index["cohort_id"].eq(cohort_id) & index["policy_hash"].eq(policy)].copy()
            units["approach"] = units["experiment_name"].fillna(units["forecast_run_id"])
            duplicates = units["approach"].duplicated(keep=False)
            units.loc[duplicates, "approach"] += " | " + units.loc[duplicates, "forecast_run_id"]
            units = units.sort_values("approach")
            state["units"] = units
            options = list(units[["approach", "forecast_run_id"]].itertuples(index=False, name=None))
            approaches.options = options
            approaches.value = tuple(units["forecast_run_id"])
            focus.options = options
            tasks = (spark.table(tables["eligibility"]).where(F.col("cohort_id") == cohort_id)
                     .select(*REVIEW_TASK).distinct().collect())
            opts = []
            for record in tasks:
                kind, metric, channel, status = values = tuple(record[c] for c in REVIEW_TASK)
                name = metric.replace("avg_", "").replace("_", " ")
                pop = "Sale" if channel == "sales" else "Rent"
                opts.append((f"{pop} | {'Eilers' if kind == 'performance' else 'Demand'} | {name}", values))
            task.options = sorted(opts, key=lambda opt: (opt[1][0] != "demand", opt[1][2] != "sales", opt[0]))
        finally:
            state["busy"] = False
        load_slice()

    cohort.observe(change_cohort, names="value")
    for widget in (task, split):
        widget.observe(load_slice, names="value")
    approaches.observe(update_approaches, names="value")
    for button in curve_switches.values():
        button.observe(update_approaches, names="value")
    for widget in (view, focus, game):
        widget.observe(render, names="value")
    display(widgets.VBox([cohort, task, split, subset, curve_box, view, focus, game, output]))
    change_cohort()
    return {"cohort": cohort, "task": task, "split": split, "approaches": approaches,
            "view": view, "focus": focus, "game": game, "curves": curve_switches, "output": output}


In [0]:
accuracy_controls = launch_accuracy_review(
    spark, review_config["outputs"], review_config["forecast"]["outputs"]
)
